# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import warnings
warnings.filterwarnings("ignore")

# Load Dataset

In [2]:
df = pd.read_csv("../data/processed/amazon_reviews_cleaned.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67807 entries, 0 to 67806
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  67807 non-null  object 
 1   product_name        61112 non-null  object 
 2   asins               67807 non-null  object 
 3   brand               67807 non-null  object 
 4   category            67807 non-null  object 
 5   keys                67807 non-null  object 
 6   manufacturer        67807 non-null  object 
 7   review_date         67807 non-null  object 
 8   reviews.dateSeen    67807 non-null  object 
 9   do_recommend        55152 non-null  float64
 10  reviews.numHelpful  55214 non-null  float64
 11  rating              67807 non-null  int64  
 12  reviews.sourceURLs  67807 non-null  object 
 13  review_text         67807 non-null  object 
 14  review_title        67788 non-null  object 
 15  reviews.username    67794 non-null  object 
 16  sent

# Text Preprocessing

In [3]:
df['clean_text'] = df['clean_text'].astype(str).str.lower()
df['clean_text'] = df['clean_text'].apply(lambda x: re.sub(r'[^a-z\s]', '', x))

# Feature Creation

In [4]:
# text features
df['review_length'] = df['clean_text'].apply(lambda x: len(x.split()))
df['num_exclamations'] = df['clean_text'].apply(lambda x: x.count('!'))
df['num_questions'] = df['clean_text'].apply(lambda x: x.count('?'))

In [5]:
# categorical features
le_brand = LabelEncoder()
df['brand_encoded'] = le_brand.fit_transform(df['brand'])

le_category = LabelEncoder()
df['category_encoded'] = le_category.fit_transform(df['category'])

# Target Variables

In [7]:
# sentiment classification
y_sentiment = df['sentiment']  # target
X_text = df['clean_text']      # text input

In [8]:
# rating predictions
y_rating = df['rating']

# Train-test split

In [9]:
# Sentiment
X_train_text, X_test_text, y_train_sent, y_test_sent = train_test_split(
    X_text, y_sentiment, test_size=0.2, random_state=42
)

# Rating
X_train_rating, X_test_rating, y_train_rating, y_test_rating = train_test_split(
    df[['review_length', 'num_exclamations', 'num_questions', 'brand_encoded', 'category_encoded']],
    y_rating, test_size=0.2, random_state=42
)

# Text Vectorization

In [10]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# Save Preprocessed Data

In [12]:
import joblib

# TF-IDF Vectorizer
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

# Preprocessed train/test splits
joblib.dump((X_train_tfidf, X_test_tfidf, y_train_sent, y_test_sent), '../models/sentiment_data.pkl')
joblib.dump((X_train_rating, X_test_rating, y_train_rating, y_test_rating), '../models/rating_data.pkl')

['../models/rating_data.pkl']